#Configuration



In [ ]:
import os
import pandas as pd
import requests
import json
import base64
import time
import glob

# DataForSEO Credentials
login = os.environ["DATAFORSEO_LOGIN"]
password = os.environ["DATAFORSEO_PASSWORD"]

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# Analysis

## Raw Data

In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

df = pd.read_csv('/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/final_merged_time_Special.csv')

if not df.empty:
    def clean_zip_code(val):
        val = str(val).strip()
        if '-' in val: val = val.split('-')[0]
        val = ''.join(filter(str.isdigit, val))
        val = val[:5]
        if val: return float(val)
        return np.nan

    if 'zip' in df.columns:
        df['zip_clean'] = df['zip'].apply(clean_zip_code)
    else:
        df['zip_clean'] = np.nan

    def map_zip_to_location(z):
        if pd.isna(z): return 'Unknown'
        try: z = int(z)
        except ValueError: return 'Unknown'
        if z == 12953: return 'Malone_NY_S'
        if z == 12983: return 'SaranacLake_NY_S'
        if 13200 <= z <= 13299: return 'Syracuse_NY_M'
        if 14200 <= z <= 14299: return 'Buffalo_NY_L'
        if (10000 <= z <= 10499) or (11000 <= z <= 11699) or (10300 <= z <= 10399): return 'NYC_NY_L'
        if z in [95501, 95502, 95503, 95534]: return 'Eureka_CA_S'
        if z == 95437: return 'FortBragg_CA_S'
        if 95350 <= z <= 95358: return 'Modesto_CA_M'
        if 94100 <= z <= 94188: return 'SanFrancisco_CA_L'
        if (90000 <= z <= 91699): return 'LA_CA_L'
        if z in [30474, 30475]: return 'Vidalia_GA_S'
        if z == 30577: return 'Toccoa_GA_S'
        if 31200 <= z <= 31299: return 'Macon_GA_M'
        if 30900 <= z <= 30999: return 'Augusta_GA_L'
        if (30300 <= z <= 30399) or (31100 <= z <= 31199): return 'Atlanta_GA_L'
        return 'Unknown'

    df['mapped_location'] = df['zip_clean'].apply(map_zip_to_location)
    df = df[df['mapped_location'] != 'Unknown'].copy()


    def get_loc_type(loc):
        if loc.endswith('_S'): return 'Small'
        if loc.endswith('_M'): return 'Mid_Size'
        if loc.endswith('_L'): return 'Large'
        return 'Unknown'
    df['Location_Type'] = df['mapped_location'].apply(get_loc_type)

    df['Location_Type'] = pd.Categorical(
        df['Location_Type'],
        categories=['Mid_Size', 'Large', 'Small'],
        ordered=False
    )


    hub_coords = {
        'NYC_NY_L': (40.7128, -74.0060), 'Buffalo_NY_L': (42.8864, -78.8784), 'Syracuse_NY_M': (43.0481, -76.1474),
        'SaranacLake_NY_S': (44.3295, -74.1313), 'Malone_NY_S': (44.8487, -74.2963),
        'LA_CA_L': (34.0522, -118.2437), 'SanFrancisco_CA_L': (37.7749, -122.4194), 'Modesto_CA_M': (37.6391, -120.9969),
        'Eureka_CA_S': (40.8021, -124.1637), 'FortBragg_CA_S': (39.4457, -123.8053),
        'Atlanta_GA_L': (33.7490, -84.3880), 'Augusta_GA_L': (33.4735, -81.9665), 'Macon_GA_M': (32.8407, -83.6324),
        'Vidalia_GA_S': (32.2177, -82.4135), 'Toccoa_GA_S': (34.5771, -83.3324)
    }

    def get_distance(row):
        loc = row['mapped_location']
        if loc not in hub_coords: return np.nan
        h_lat, h_lon = hub_coords[loc]
        try:
            lon1, lat1, lon2, lat2 = map(radians, [float(row['longitude']), float(row['latitude']), h_lon, h_lat])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return c * 3958.8
        except: return np.nan

    if 'latitude' in df.columns:
        df['distance_to_hub'] = df.apply(get_distance, axis=1)

    df['zip_clinic_count'] = df.groupby('zip_clean')['zip_clean'].transform('count')

    def calc_peer_dist(sub):
        if len(sub) < 2: return np.zeros(len(sub))
        lats = np.radians(sub['latitude'].values)
        lons = np.radians(sub['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        dists = c * 3958.8
        return np.sum(dists, axis=1) / (len(sub) - 1)

    coords = df[['latitude', 'longitude', 'zip_clean']].dropna()
    peer_dist_map = {}
    for z, group in coords.groupby('zip_clean'):
        if len(group) > 1:
            dists = calc_peer_dist(group)
            for idx, val in zip(group.index, dists):
                peer_dist_map[idx] = val
        else:
            for idx in group.index:
                peer_dist_map[idx] = 0.0

    df['avg_peer_dist'] = df.index.map(peer_dist_map).fillna(0)
    df['avg_peer_dist'] = df['avg_peer_dist'].replace(0.0, np.nan)

    df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')

    df_low = df[df['rating_value'] <= 3.0].copy()
    peer_dist_low_map = {}
    if not df_low.empty:
        coords_low = df_low[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_low.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_low_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_low_map[idx] = 0.0

    df['avg_dist_to_1star_peers'] = df.index.map(peer_dist_low_map)
    df['avg_dist_to_1star_peers'] = df['avg_dist_to_1star_peers'].replace(0.0, np.nan)

    df_other = df[df['rating_value'] > 3.0].copy()
    peer_dist_other_map = {}
    if not df_other.empty:
        coords_other = df_other[['latitude', 'longitude', 'zip_clean']].dropna()
        for z, group in coords_other.groupby('zip_clean'):
            if len(group) > 1:
                dists = calc_peer_dist(group)
                for idx, val in zip(group.index, dists):
                    peer_dist_other_map[idx] = val
            else:
                for idx in group.index:
                    peer_dist_other_map[idx] = 0.0

    df['avg_dist_to_other_peers'] = df.index.map(peer_dist_other_map)
    df['avg_dist_to_other_peers'] = df['avg_dist_to_other_peers'].replace(0.0, np.nan)

    df['log_votes'] = np.log1p(pd.to_numeric(df['votes_count'], errors='coerce').fillna(0))
    df['is_low_rating'] = np.where(df['rating_value'] <= 3, 1, 0)

    def get_state(loc):
        if '_NY_' in loc: return 'NY'
        if '_CA_' in loc: return 'CA'
        if '_GA_' in loc: return 'GA'
        return 'Other'
    df['State'] = df['mapped_location'].apply(get_state)

    reg_df = df.dropna(subset=[
        'rating_value', 'distance_to_hub', 'log_votes',
        'zip_clinic_count', 'State', 'Location_Type'
    ]).copy()

    print(f"N = {len(reg_df)}")

In [ ]:
import pandas as pd

df_rated = df.dropna(subset=['rating_value']).copy()

low_group = df_rated[df_rated['rating_value'] <= 3.0]
high_group = df_rated[df_rated['rating_value'] > 3.0]

low_total = len(low_group)
low_with_time = low_group['start_date'].notna().sum()
low_prop = (low_with_time / low_total) if low_total > 0 else 0

high_total = len(high_group)
high_with_time = high_group['start_date'].notna().sum()
high_prop = (high_with_time / high_total) if high_total > 0 else 0

print(f"N= {len(df_rated)}")
print("-" * 40)
print(f"N_low= {low_total}")
print(f"N_low with start_date= {low_with_time}")
print(f"Coverage: {low_prop:.2%}")
print("-" * 40)
print(f"N_high= {high_total}")
print(f"N_high with start_date= {high_with_time}")
print(f"Coverage: {high_prop:.2%}")

low_missing_date_df = low_group[low_group['start_date'].isna()]
low_missing_date_clinics = low_missing_date_df['title'].tolist()

print("-" * 40)
print(f"Clinics in low group missing start_date (N={len(low_missing_date_clinics)}):")
for clinic in low_missing_date_clinics:
    print(clinic)

评分少且一般是连锁或几家店叫一个名字，网站无法精准对应

In [ ]:
low_group = df[df['rating_value'] <= 3.0]
high_group = df[df['rating_value'] > 3.0]

mean_hub_low = low_group['distance_to_hub'].mean()
mean_hub_high = high_group['distance_to_hub'].mean()
mean_hub_all = df['distance_to_hub'].mean()

mean_peer_all_low = low_group['avg_peer_dist'].mean()
mean_peer_all_high = high_group['avg_peer_dist'].mean()
mean_peer_all = df['avg_peer_dist'].mean()

mean_peer_low_low = low_group['avg_dist_to_1star_peers'].mean()
mean_peer_high_high = high_group['avg_dist_to_other_peers'].mean()

print(f"Avg Dist to Hub:")
print(f"   - Low Rate Clinics:  {mean_hub_low:.2f} miles")
print(f"   - High Rate Clinics: {mean_hub_high:.2f} miles")
print(f"   - All Clinics:       {mean_hub_all:.2f} miles")

print(f"\nAvg Dist to Peers (All peers in zip):")
print(f"   - Low Rate Clinics:  {mean_peer_all_low:.2f} miles")
print(f"   - High Rate Clinics: {mean_peer_all_high:.2f} miles")
print(f"   - All Clinics:       {mean_peer_all:.2f} miles")

print(f"\nAvg Dist to Same-Tier Peers:")
print(f"   - Low Rate to Low Rate:   {mean_peer_low_low:.2f} miles")
print(f"   - High Rate to High Rate: {mean_peer_high_high:.2f} miles")

In [ ]:
stats = []
for loc, group in df.groupby('mapped_location'):
    total_clinics = len(group)
    low_clinics = len(group[group['rating_value'] <= 3.0])
    low_pct = low_clinics / total_clinics if total_clinics > 0 else 0
    avg_low_to_low = group[group['rating_value'] <= 3.0]['avg_dist_to_1star_peers'].mean()

    stats.append({
        'Region': loc,
        'Total_Clinics': total_clinics,
        'Low_Rate_Clinics': low_clinics,
        'Low_Rate_Pct': low_pct,
        'Avg_Dist_Low_to_Low': avg_low_to_low
    })

res_df = pd.DataFrame(stats).sort_values('Low_Rate_Clinics', ascending=False)
print(res_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree

df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
df_valid = df.dropna(subset=['latitude', 'longitude', 'rating_value']).copy()

df_low = df_valid[df_valid['rating_value'] <= 3.0].copy()

coords_low = np.radians(df_low[['latitude', 'longitude']].values)
coords_all = np.radians(df_valid[['latitude', 'longitude']].values)

db = DBSCAN(eps=2.0/3958.8, min_samples=2, metric='haversine').fit(coords_low)
df_low['cluster'] = db.labels_
clusters = df_low[df_low['cluster'] != -1]

results = []
tree_all = BallTree(coords_all, metric='haversine')

for c_id, group in clusters.groupby('cluster'):
    center_lat = group['latitude'].mean()
    center_lon = group['longitude'].mean()
    n_low = len(group)

    if n_low > 1:
        lats = np.radians(group['latitude'].values)
        lons = np.radians(group['longitude'].values)
        dlat = lats[:, None] - lats[None, :]
        dlon = lons[:, None] - lons[None, :]
        a = np.sin(dlat/2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon/2)**2
        dists = 2 * np.arcsin(np.sqrt(a)) * 3958.8
        avg_dist = np.sum(dists) / (n_low * (n_low - 1))
    else:
        avg_dist = 0

    centroid_rad = np.radians([[center_lat, center_lon]])

    n_total_2mi = tree_all.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_total_5mi = tree_all.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    results.append({
        'Cluster_ID': c_id,
        'Center_Lat': round(center_lat, 4),
        'Center_Lon': round(center_lon, 4),
        'Low_Clinics_Count': n_low,
        'Avg_Dist_Low_to_Low': round(avg_dist, 4),
        'Total_Clinics_2mi': n_total_2mi,
        'Low_Pct_2mi': round(n_low / n_total_2mi, 4) if n_total_2mi > 0 else 0,
        'Total_Clinics_5mi': n_total_5mi,
        'Low_Pct_5mi': round(n_low / n_total_5mi, 4) if n_total_5mi > 0 else 0
    })

res_df = pd.DataFrame(results).sort_values(by=['Low_Pct_2mi', 'Low_Clinics_Count'], ascending=[False, False])
print(res_df.to_string(index=False))

In [ ]:
df['rating_value'] = pd.to_numeric(df['rating_value'], errors='coerce')
df_valid = df.dropna(subset=['latitude', 'longitude', 'rating_value']).copy()

df_high = df_valid[df_valid['rating_value'] > 4.8].copy()

coords_high = np.radians(df_high[['latitude', 'longitude']].values)
coords_all = np.radians(df_valid[['latitude', 'longitude']].values)

db = DBSCAN(eps=2.0/3958.8, min_samples=2, metric='haversine').fit(coords_high)
df_high['cluster'] = db.labels_
clusters = df_high[df_high['cluster'] != -1]

results = []
tree_all = BallTree(coords_all, metric='haversine')
tree_high = BallTree(coords_high, metric='haversine')

for c_id, group in clusters.groupby('cluster'):
    center_lat = group['latitude'].mean()
    center_lon = group['longitude'].mean()

    centroid_rad = np.radians([[center_lat, center_lon]])

    n_total_2mi = tree_all.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_total_5mi = tree_all.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    n_high_2mi = tree_high.query_radius(centroid_rad, r=2.0/3958.8, count_only=True)[0]
    n_high_5mi = tree_high.query_radius(centroid_rad, r=5.0/3958.8, count_only=True)[0]

    if n_total_2mi < 10:
        continue

    high_pct_2mi = n_high_2mi / n_total_2mi if n_total_2mi > 0 else 0
    high_pct_5mi = n_high_5mi / n_total_5mi if n_total_5mi > 0 else 0

    results.append({
        'Cluster_ID': c_id,
        'Center_Lat': round(center_lat, 4),
        'Center_Lon': round(center_lon, 4),
        'High_Clinics_2mi': n_high_2mi,
        'Total_Clinics_2mi': n_total_2mi,
        'High_Pct_2mi': high_pct_2mi,
        'Total_Clinics_5mi': n_total_5mi,
        'High_Pct_5mi': high_pct_5mi
    })


res_df = pd.DataFrame(results).sort_values(by=['High_Pct_2mi', 'High_Clinics_2mi'], ascending=[False, False])


res_df['High_Pct_2mi'] = res_df['High_Pct_2mi'].apply(lambda x: f"{x:.2%}")
res_df['High_Pct_5mi'] = res_df['High_Pct_5mi'].apply(lambda x: f"{x:.2%}")

print(res_df.head(25).to_string(index=False))

=========To hub==========
* 90502: Torrance, CA
* 30474: Vidalia, GA (small city)
* 30324: Atlanta, GA (Buckhead/Lindbergh area)
* 90001: Los Angeles, CA (South LA/Florence)
* 30331: Atlanta, GA (Southwest Atlanta)

=========To peer=========
* 14226: Amherst/Williamsville, NY (Buffalo suburb)
* 31201: Macon, GA (Downtown)
* 14203: Buffalo, NY (Downtown)
* 10038: New York, NY (Manhattan - Financial District)
* 10007: New York, NY (Manhattan - Tribeca/Civic Center)


## Regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')

df_reg = df.dropna(subset=['rating_value', 'latitude', 'longitude', 'start_date', 'votes_count', 'search_location']).copy()

df_reg['entry_year'] = pd.to_datetime(df_reg['start_date'], errors='coerce').dt.year
df_reg = df_reg.dropna(subset=['entry_year'])
df_reg['entry_year'] = df_reg['entry_year'].astype(int)

df_reg['clinic_age'] = 2024 - df_reg['entry_year']
df_reg['log_votes'] = np.log1p(df_reg['votes_count'].fillna(0))

coords_rad = np.radians(df_reg[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')
radius_2mi = 2.0 / 3958.8

# find in 2 mile all neighbors index
indices_2mi = tree.query_radius(coords_rad, r=radius_2mi)

lag_entry_2mi_count = []
density_2mi_total = []

for i, neighbors in enumerate(indices_2mi):
    current_year = df_reg.iloc[i]['entry_year']

    # 2-mile neighbor enrty year
    neighbor_years = df_reg.iloc[neighbors]['entry_year'].values

    # Lag Entry 2mi: last year num of entry in 2 miles
    lag_count = np.sum(neighbor_years == (current_year - 1))
    lag_entry_2mi_count.append(lag_count)

    # Density 2mi
    density_count = np.sum(neighbor_years <= current_year) - 1
    density_2mi_total.append(density_count if density_count > 0 else 0)


df_reg['lag_entry_shock_2mi'] = lag_entry_2mi_count
df_reg['lag_entry_shock_2mi_dummy'] = (df_reg['lag_entry_shock_2mi'] > 0).astype(int)
df_reg['log_density_2mi_total'] = np.log1p(density_2mi_total)

### Panel

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')

print("Build Panel Data")

df_reg = df.dropna(subset=['rating_value', 'latitude', 'longitude', 'start_date', 'mapped_location']).copy()
df_reg['entry_year'] = pd.to_datetime(df_reg['start_date'], errors='coerce').dt.year
df_reg = df_reg.dropna(subset=['entry_year'])
df_reg['entry_year'] = df_reg['entry_year'].astype(int)

df_reg = df_reg.reset_index(drop=True)
df_reg['clinic_id'] = ["C_" + str(i) for i in range(len(df_reg))]

# 2-Mile neighbors
coords_rad = np.radians(df_reg[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')
indices_2mi = tree.query_radius(coords_rad, r=2.0 / 3958.8)

# Expand to panel format
panel_records = []
for i, row in df_reg.iterrows():
    entry_y = int(row['entry_year'])
    #  entry years of all competitors within 2 miles
    neighbor_indices = indices_2mi[i]
    neighbor_years = df_reg.iloc[neighbor_indices]['entry_year'].values
    neighbor_titles = df_reg.iloc[neighbor_indices]['title'].values
    neighbor_ids = df_reg.iloc[neighbor_indices]['clinic_id'].values

    for current_year in range(entry_y, 2026):
        # dynamic spatial density and shocks
        density_count_t = max(0, np.sum(neighbor_years <= current_year) - 1)
        lag_count_t = np.sum(neighbor_years == (current_year - 1))

        shock_mask = (neighbor_years == (current_year - 1)) & (neighbor_ids != row['clinic_id'])
        shock_names = ", ".join(neighbor_titles[shock_mask])

        panel_records.append({
            'clinic_id': row['clinic_id'],
            'title': row['title'],
            'zip_clean': row['zip_clean'],
            'year': current_year,
            'clinic_age': current_year - entry_y,
            'log_density_2mi_total': np.log1p(density_count_t),
            'lag_entry_shock_2mi_count': lag_count_t,
            'lag_entry_shock_2mi_names': shock_names,
            'lag_entry_shock_2mi_dummy': 1 if lag_count_t > 0 else 0,
            'log_lag_entry_shock_2mi': np.log1p(lag_count_t),
            'distance_to_hub': row['distance_to_hub'],
            'mapped_location': row['mapped_location'],
        })

df_panel = pd.DataFrame(panel_records)

# Process all review for dynamic ratings and votes
REVIEW_NAME_COL = 'shop_title'
REVIEW_ZIP_COL = 'shop_zip'
REVIEW_DATE_COL = 'date'
REVIEW_RATING_COL = 'rating'

chunk_size = 10000
yearly_stats_chunks = []
cols_to_use = [REVIEW_NAME_COL, REVIEW_ZIP_COL, REVIEW_DATE_COL, REVIEW_RATING_COL]

print("Chunking and aggregating reviews...")
try:
    for chunk in pd.read_csv("/content/drive/MyDrive/RA/health_care/dentist_LMS_keywords/output/final/all_reviews_detailed_Special.csv", chunksize=chunk_size, usecols=cols_to_use):
        # match zip code
        chunk[REVIEW_ZIP_COL] = chunk[REVIEW_ZIP_COL].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
        chunk['review_year'] = pd.to_datetime(chunk[REVIEW_DATE_COL], errors='coerce').dt.year
        chunk = chunk.dropna(subset=['review_year', REVIEW_RATING_COL])
        chunk['review_year'] = chunk['review_year'].astype(int)

        # aggregate count and sum per year
        stats_df = chunk.groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year'])[REVIEW_RATING_COL].agg(['count', 'sum']).reset_index()
        yearly_stats_chunks.append(stats_df)

    final_yearly_stats = pd.concat(yearly_stats_chunks).groupby([REVIEW_NAME_COL, REVIEW_ZIP_COL, 'review_year']).sum().reset_index()
    final_yearly_stats = final_yearly_stats.rename(columns={REVIEW_NAME_COL: 'title', REVIEW_ZIP_COL: 'zip_str', 'count': 'new_count', 'sum': 'new_stars'})
except Exception as e:
    print(f"   [Warning] Failed to read reviews: {e}")

# Merge reviews into panel
df_panel['zip_str'] = df_panel['zip_clean'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

df_panel = df_panel.merge(final_yearly_stats, left_on=['title', 'zip_str', 'year'], right_on=['title', 'zip_str', 'review_year'], how='left')
df_panel['new_count'] = df_panel['new_count'].fillna(0)
df_panel['new_stars'] = df_panel['new_stars'].fillna(0)

# sort
df_panel = df_panel.sort_values(by=['clinic_id', 'year'])
df_panel['cumulative_votes'] = df_panel.groupby('clinic_id')['new_count'].cumsum()
df_panel['cumulative_stars'] = df_panel.groupby('clinic_id')['new_stars'].cumsum()

# dynamic vote
df_panel['dynamic_rating'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_stars'] / df_panel['cumulative_votes'], np.nan)
df_panel['log_votes_dynamic'] = np.log1p(df_panel['cumulative_votes'])

# drop early years where the clinic hadn't received first review
df_panel_valid = df_panel.dropna(subset=['dynamic_rating']).reset_index(drop=True)
print(f"Dynamic Panel successfully constructed! Observations: {len(df_panel_valid)}")




In [ ]:
import pandas as pd
import numpy as np

print("Panel Data Check")

print("\nData Info")
print(f"Observations: {len(df_panel_valid)}")
print(f"Unique clinic(N): {df_panel_valid['clinic_id'].nunique()}")
print(f"Time range(T): {df_panel_valid['year'].min()} to {df_panel_valid['year'].max()}")

# Check missing data
print("\nCheck missing data")
core_vars = ['dynamic_rating', 'cumulative_votes', 'log_votes_dynamic', 'lag_entry_shock_2mi_count', 'log_density_2mi_total']
print(df_panel_valid[core_vars].isnull().sum())

# Stat
print("\nStat")
print(df_panel_valid[core_vars].describe().round(3))



In [ ]:
print("Panel Data Check")

print("\nData Info")
print(f"Observations: {len(df_panel_valid)}")
print(f"Unique clinic(N): {df_panel_valid['clinic_id'].nunique()}")
print(f"Time range(T): {df_panel_valid['year'].min()} to {df_panel_valid['year'].max()}")

# Check missing data
print("\nCheck missing data")
core_vars = ['dynamic_rating', 'cumulative_votes', 'log_votes_dynamic', 'lag_entry_shock_2mi_count', 'log_density_2mi_total']
print(df_panel_valid[core_vars].isnull().sum())

# Stat by Year
print("\nStat by Year")
# Loop through each year in chronological order
for year in sorted(df_panel_valid['year'].unique()):
    print(f"\n{'-'*50}")
    print(f"Summary Statistics for Year: {year}")
    print(f"{'-'*50}")
    # Filter data for the specific year and print the describe() table
    year_data = df_panel_valid[df_panel_valid['year'] == year][core_vars]
    print(year_data.describe().round(3))

In [ ]:
# random clinic
print("\nRandom clinic info")
sample_clinic = np.random.choice(df_panel_valid['clinic_id'].unique())
full_sample_data = df_panel_valid[df_panel_valid['clinic_id'] == sample_clinic]

clinic_title = full_sample_data['title'].iloc[0]
clinic_zip = full_sample_data['zip_clean'].iloc[0]

sample_data = df_panel_valid[df_panel_valid['clinic_id'] == sample_clinic][
    ['year', 'new_count', 'cumulative_votes', 'dynamic_rating', 'lag_entry_shock_2mi_count', 'log_density_2mi_total', 'lag_entry_shock_2mi_names']
]
print(f"Sample clinic: {sample_clinic}")
print(f"Clinic Name: {clinic_title} (Zip: {clinic_zip})")
print(sample_data.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf

# Lag Entry Shock Comparison
print("\nLag Entry Shock Comparison")

# Using the Geographic FE framework as it optimally balances macro controls and time dynamics
formula_comp = "dynamic_rating ~ {} + log_density_2mi_total + clinic_age + log_votes_dynamic + distance_to_hub + C(mapped_location)"

mod_dummy = smf.ols(formula_comp.format("lag_entry_shock_2mi_dummy"), data=df_panel_valid).fit()
mod_count = smf.ols(formula_comp.format("lag_entry_shock_2mi_count"), data=df_panel_valid).fit()
mod_log   = smf.ols(formula_comp.format("log_lag_entry_shock_2mi"), data=df_panel_valid).fit()

results = []
for name, mod, var in [('1. Dummy (0/1)', mod_dummy, 'lag_entry_shock_2mi_dummy'),
                       ('2. Cont. Count', mod_count, 'lag_entry_shock_2mi_count'),
                       ('3. Log(Count)', mod_log, 'log_lag_entry_shock_2mi')]:
    results.append({
        'Shock Form': name,
        'AIC': round(mod.aic, 2),
        'BIC': round(mod.bic, 2),
        'R-squared': round(mod.rsquared, 4),
        'Coefficient': round(mod.params[var], 5),
        'P-value': f"{mod.pvalues[var]:.3e}"
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))


In [ ]:
# Geographic fixed effect
print("Geographic fixed effect")

formula_geo_pooled = "dynamic_rating ~ log_lag_entry_shock_2mi + log_density_2mi_total + clinic_age + log_votes_dynamic + distance_to_hub"
mod_geo_pooled = smf.ols(formula_geo_pooled, data=df_panel_valid).fit()

# F-Test
mod_geo_fe = smf.ols(formula_geo_pooled + " + C(mapped_location)", data=df_panel_valid).fit()
f_test_geo = mod_geo_fe.compare_f_test(mod_geo_pooled)
print(f"[Geo F-Test] P-value: {f_test_geo[1]:.4e} -> {'Reject the null hypothesis. Significant geographic fixed effects exist.' if f_test_geo[1]<0.05 else 'Cannot reject the null hypothesis. Geographic fixed effects not significant.'}")

# Hausman Test
try:
    mod_geo_re = smf.mixedlm(formula_geo_pooled, data=df_panel_valid, groups=df_panel_valid['mapped_location']).fit()
    cv_geo = [v for v in mod_geo_re.params.index if v in mod_geo_fe.params.index and v != 'Intercept']
    diff_geo = mod_geo_fe.params[cv_geo] - mod_geo_re.params[cv_geo]
    v_diff_geo = mod_geo_fe.cov_params().loc[cv_geo, cv_geo] - mod_geo_re.cov_params().loc[cv_geo, cv_geo]

    try:
        chi2_geo = np.dot(diff_geo.T, np.dot(np.linalg.inv(v_diff_geo), diff_geo))
        if chi2_geo < 0:
            print(f"[Geo Hausman] Chi2: {chi2_geo:.4f} (Invalid: Non-positive definite matrix. )")
        else:
            pval_geo = 1 - stats.chi2.cdf(chi2_geo, len(cv_geo))
            print(f"[Geo Hausman] Chi2: {chi2_geo:.4f}, P-value: {pval_geo:.4e} -> {'Use FE' if pval_geo<0.05 else 'Use RE'}")
    except np.linalg.LinAlgError:
        print("[Geo Hausman Warning] Singular matrix detected. Attempting pseudo-inverse...")
        chi2_geo_pinv = np.dot(diff_geo.T, np.dot(np.linalg.pinv(v_diff_geo), diff_geo))
        if chi2_geo_pinv < 0:
            print(f"[Geo Hausman Pseudo-Inv] Chi2: {chi2_geo_pinv:.4f} (Invalid: Non-positive definite. )")
        else:
            pval_geo_pinv = 1 - stats.chi2.cdf(chi2_geo_pinv, len(cv_geo))
            print(f"[Geo Hausman Pseudo-Inv] Chi2: {chi2_geo_pinv:.4f}, P-value: {pval_geo_pinv:.4e} -> {'Use FE' if pval_geo_pinv<0.05 else 'Use RE'}")
except Exception as e:
    print(f"[Geo Hausman Error] Computation failed: {e}")

# Mundlak
df_panel_valid['mean_density_geo'] = df_panel_valid.groupby('mapped_location')['log_density_2mi_total'].transform('mean')
df_panel_valid['mean_age_geo'] = df_panel_valid.groupby('mapped_location')['clinic_age'].transform('mean')
try:
    mod_geo_mundlak = smf.mixedlm(formula_geo_pooled + " + mean_density_geo + mean_age_geo", data=df_panel_valid, groups=df_panel_valid['mapped_location']).fit()
    wald_geo = mod_geo_mundlak.wald_test("mean_density_geo = 0, mean_age_geo = 0")
    print(f"[Geo Mundlak] P-value: {wald_geo.pvalue.item():.4e} -> {'Use FE' if wald_geo.pvalue.item()<0.05 else 'Use RE'}")
except Exception as e:
    print(f"[Geo Mundlak] Encountered convergence issue: {e}")


# Individual Fixed Effect Tests
print("\nIndividual fixed effect")
print("'distance_to_hub' and 'mapped_location' are excluded")

formula_ind_pooled = "dynamic_rating ~ log_lag_entry_shock_2mi + log_density_2mi_total + clinic_age + log_votes_dynamic"
mod_ind_pooled = smf.ols(formula_ind_pooled, data=df_panel_valid).fit()

# F-Test
try:
    mod_ind_fe = smf.ols(formula_ind_pooled + " + C(clinic_id)", data=df_panel_valid).fit()
    f_test_ind = mod_ind_fe.compare_f_test(mod_ind_pooled)
    print(f"[Ind F-Test] P-value: {f_test_ind[1]:.4e} -> {'Reject the null hypothesis. Significant individual fixed effects exist.' if f_test_ind[1]<0.05 else 'Cannot reject the null hypothesis. Individual fixed effects not significant.'}")
except Exception as e:
    print(f"[Ind F-Test] Computation failed: {e}")

# Hausman Test
try:
    mod_ind_re = smf.mixedlm(formula_ind_pooled, data=df_panel_valid, groups=df_panel_valid['clinic_id']).fit()
    cv_ind = [v for v in mod_ind_re.params.index if v in mod_ind_fe.params.index and v != 'Intercept']
    diff_ind = mod_ind_fe.params[cv_ind] - mod_ind_re.params[cv_ind]
    v_diff_ind = mod_ind_fe.cov_params().loc[cv_ind, cv_ind] - mod_ind_re.cov_params().loc[cv_ind, cv_ind]

    try:
        chi2_ind = np.dot(diff_ind.T, np.dot(np.linalg.inv(v_diff_ind), diff_ind))
        if chi2_ind < 0:
            print(f"[Ind Hausman] Chi2: {chi2_ind:.4f} (Invalid: Non-positive definite matrix)")
        else:
            pval_ind = 1 - stats.chi2.cdf(chi2_ind, len(cv_ind))
            print(f"[Ind Hausman] Chi2: {chi2_ind:.4f}, P-value: {pval_ind:.4e} -> {'use FE' if pval_ind<0.05 else 'use RE'}")
    except np.linalg.LinAlgError:
        print("[Ind Hausman Warning] Singular matrix detected. Attempting pseudo-inverse...")
        chi2_ind_pinv = np.dot(diff_ind.T, np.dot(np.linalg.pinv(v_diff_ind), diff_ind))
        if chi2_ind_pinv < 0:
            print(f"[Ind Hausman Pseudo-Inv] Chi2: {chi2_ind_pinv:.4f} (Invalid: Non-positive definite)")
        else:
            pval_ind_pinv = 1 - stats.chi2.cdf(chi2_ind_pinv, len(cv_ind))
            print(f"[Ind Hausman Pseudo-Inv] Chi2: {chi2_ind_pinv:.4f}, P-value: {pval_ind_pinv:.4e} -> {'use FE' if pval_ind_pinv<0.05 else 'use RE'}")
except Exception as e:
    print(f"[Ind Hausman Error] Computation or convergence failed: {e}")

# Mundlak
try:
    df_panel_valid['mean_density_ind'] = df_panel_valid.groupby('clinic_id')['log_density_2mi_total'].transform('mean')
    df_panel_valid['mean_age_ind'] = df_panel_valid.groupby('clinic_id')['clinic_age'].transform('mean')

    mod_ind_mundlak = smf.mixedlm(formula_ind_pooled + " + mean_density_ind + mean_age_ind", data=df_panel_valid, groups=df_panel_valid['clinic_id']).fit()
    wald_ind = mod_ind_mundlak.wald_test("mean_density_ind = 0, mean_age_ind = 0")
    print(f"[Ind Mundlak] P-value: {wald_ind.pvalue.item():.4e} -> {'use FE' if wald_ind.pvalue.item()<0.05 else 'use RE'}")
except Exception as e:
    print(f"[Ind Mundlak] Failed to converge due to extreme dimensions: {e}")

In [ ]:
!pip install linearmodels

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

def get_loc_type(loc):
    loc_str = str(loc)
    if loc_str.endswith('_S'): return 'Small'
    if loc_str.endswith('_M'): return 'Mid_Size'
    if loc_str.endswith('_L'): return 'Large'
    return 'Unknown'

df_panel_valid['Location_Type'] = df_panel_valid['mapped_location'].apply(get_loc_type)

df_panel_valid['Location_Type'] = pd.Categorical(
    df_panel_valid['Location_Type'],
    categories=['Mid_Size', 'Large', 'Small'],
    ordered=False
)



print("Panel OLS (Individual Fixed Effects)")

# Entity = clinic_id, Time = year
if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])


formula_fe = """
    dynamic_rating ~ log_lag_entry_shock_2mi
                   + log_density_2mi_total
                   + clinic_age
                   + log_votes_dynamic
                   + EntityEffects
"""
mod_fe = PanelOLS.from_formula(formula_fe, data=df_panel_valid, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)

print(mod_fe.summary)


print("Survival Model (<= 3.5)")

df_surv = df_panel_valid.reset_index()

df_surv['event_sub_4'] = (df_surv['dynamic_rating'] <= 3.5).astype(int)

first_failure = df_surv[df_surv['event_sub_4'] == 1].groupby('clinic_id')['year'].min().reset_index()
first_failure.rename(columns={'year': 'failure_year'}, inplace=True)

df_surv = df_surv.merge(first_failure, on='clinic_id', how='left')
df_surv_strict = df_surv[(df_surv['failure_year'].isna()) | (df_surv['year'] <= df_surv['failure_year'])].copy()


formula_surv = """
    event_sub_4 ~ log_lag_entry_shock_2mi
                + log_density_2mi_total
                + clinic_age
                + log_votes_dynamic
                + distance_to_hub
                + C(Location_Type)
"""
mod_survival = smf.logit(formula_surv, data=df_surv_strict).fit(cov_type='cluster', cov_kwds={'groups': df_surv_strict['clinic_id']})

print(mod_survival.summary())

In [ ]:
# Strong competitor
print("Strong competitor")

if isinstance(df_panel_valid.index, pd.MultiIndex):
    df_temp = df_panel_valid.reset_index()
else:
    df_temp = df_panel_valid.copy()

df_temp['clinic_id'] = df_temp['clinic_id'].astype(str)
df_temp['year'] = df_temp['year'].astype(int)

mean_votes_by_year = df_temp.groupby('year')['new_count'].transform('mean')
df_temp['is_strong_entity'] = (
    (df_temp['dynamic_rating'] >= 4.9) &
    (df_temp['new_count'] > mean_votes_by_year)
).astype(int)


strong_lookup = df_temp.set_index(['clinic_id', 'year'])['is_strong_entity'].to_dict()


if 'clinic_id' not in df_reg.columns:
    df_reg_clean = df_reg.reset_index()
else:
    df_reg_clean = df_reg.copy()
df_reg_clean['clinic_id'] = df_reg_clean['clinic_id'].astype(str)

strong_shock_records = []

for i, row in df_reg_clean.iterrows():
    cid = str(row['clinic_id'])
    entry_y = int(row['entry_year'])
    neighbor_indices = indices_2mi[i]
    neighbor_ids = [str(df_reg_clean.iloc[n_idx]['clinic_id']) for n_idx in neighbor_indices if str(df_reg_clean.iloc[n_idx]['clinic_id']) != cid]

    for current_year in range(entry_y, 2025):
        lag_year = current_year - 1
        strong_count = sum(1 for n_id in neighbor_ids if (n_id, lag_year) in strong_lookup and strong_lookup[(n_id, lag_year)] == 1)

        strong_shock_records.append({
            'clinic_id': cid,
            'year': current_year,
            'strong_lag_shock_count': strong_count,
            'strong_lag_shock_dummy': 1 if strong_count > 0 else 0
        })

df_strong_shock = pd.DataFrame(strong_shock_records)

df_panel_hetero = df_temp.merge(df_strong_shock, on=['clinic_id', 'year'], how='left')
df_panel_hetero['strong_lag_shock_count'] = df_panel_hetero['strong_lag_shock_count'].fillna(0)
df_panel_hetero['strong_lag_shock_dummy'] = df_panel_hetero['strong_lag_shock_dummy'].fillna(0)

# OLS
df_panel_hetero = df_panel_hetero.set_index(['clinic_id', 'year'])

print("\nPanel OLS")
formula_hetero_ols = """
    dynamic_rating ~ log_lag_entry_shock_2mi
                   + strong_lag_shock_dummy
                   + log_density_2mi_total
                   + clinic_age
                   + log_votes_dynamic
                   + EntityEffects
"""
mod_hetero_ols = PanelOLS.from_formula(formula_hetero_ols, data=df_panel_hetero, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
print(mod_hetero_ols.summary)

# Survival Model (<= 3.5)
print("\nSurvival Model (<= 3.5)")


df_surv_hetero = df_panel_hetero.reset_index()
df_surv_hetero['event_sub_4'] = (df_surv_hetero['dynamic_rating'] <= 3.5).astype(int)


first_fail_h = df_surv_hetero[df_surv_hetero['event_sub_4'] == 1].groupby('clinic_id')['year'].min().reset_index()
first_fail_h.rename(columns={'year': 'failure_year'}, inplace=True)


df_surv_h_merged = df_surv_hetero.merge(first_fail_h, on='clinic_id', how='left')
df_surv_h_strict = df_surv_h_merged[(df_surv_h_merged['failure_year'].isna()) | (df_surv_h_merged['year'] <= df_surv_h_merged['failure_year'])].copy()


formula_hetero_surv = """
    event_sub_4 ~ log_lag_entry_shock_2mi
                + strong_lag_shock_dummy
                + log_density_2mi_total
                + clinic_age
                + log_votes_dynamic
                + distance_to_hub
                + C(Location_Type)
"""

try:
    mod_hetero_surv = smf.logit(formula_hetero_surv, data=df_surv_h_strict).fit(
        cov_type='cluster',
        cov_kwds={'groups': df_surv_h_strict['clinic_id']},
        disp=0
    )
    print(mod_hetero_surv.summary())
except Exception as e:
    print(f"Survival Model fitting failed: {e}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

if isinstance(df_panel_hetero.index, pd.MultiIndex):
    df_plot_data = df_panel_hetero.reset_index()
else:
    df_plot_data = df_panel_hetero.copy()

def assign_group(row):
    if row['log_lag_entry_shock_2mi'] == 0:
        return 'Group A: No Entry'
    elif row['strong_lag_shock_dummy'] == 0:
        return 'Group B: Weak Entry Only'
    elif row['strong_lag_shock_dummy'] == 1:
        return 'Group C: Strong Entry (4.9+)'
    else:
        return 'Other'

df_plot_data['Comp_Group'] = df_plot_data.apply(assign_group, axis=1)

sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})
palette = {'Group A: No Entry': '#95a5a6',
           'Group B: Weak Entry Only': '#3498db',
           'Group C: Strong Entry (4.9+)': '#e74c3c'}


g = sns.FacetGrid(df_plot_data, col="Comp_Group", hue="Comp_Group",
                  palette=palette, height=4, aspect=1.2, col_wrap=2)

g.map(sns.kdeplot, "dynamic_rating", fill=True, alpha=0.6, bw_adjust=1.2)
g.map(sns.kdeplot, "dynamic_rating", color="white", lw=2, bw_adjust=1.2)

def annotate_plot(data, **kwargs):
    ax = plt.gca()
    mean_val = data['dynamic_rating'].mean()
    ax.axvline(mean_val, color='black', linestyle='--', lw=1.5)
    ax.text(mean_val*1.02, ax.get_ylim()[1]*0.8, f'Mean: {mean_val:.2f}',
            fontsize=11, fontweight='bold')
    ax.axvline(3.5, color='#c0392b', linestyle='-', lw=2)
    ax.set_xlim(2.5, 5.1)

g.map_dataframe(annotate_plot)

g.set_titles(col_template="{col_name}", size=13, fontweight='bold')
g.set_axis_labels("Dynamic Rating", "Density")
g.fig.suptitle('Density across Competition Shocks',
               fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()
plt.show()

# Histogram
fig_hist, axes_hist = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
axes_hist = axes_hist.flatten()

for i, (group_name, group_data) in enumerate(df_plot_data.groupby('Comp_Group')):
    if group_name == 'Other':
        continue
    ax = axes_hist[i]
    sns.histplot(group_data['dynamic_rating'], ax=ax, kde=False, stat='density',
                 color=palette[group_name], edgecolor='black', bins=10)
    ax.set_title(group_name, size=13, fontweight='bold')
    ax.set_xlabel("Dynamic Rating")
    ax.set_ylabel("Density")
    ax.set_xlim(2.5, 5.1)

    # Add mean and 3.5 rating line
    mean_val = group_data['dynamic_rating'].mean()
    ax.axvline(mean_val, color='black', linestyle='--', lw=1.5)
    ax.text(mean_val*1.02, ax.get_ylim()[1]*0.8, f'Mean: {mean_val:.2f}',
            fontsize=11, fontweight='bold')
    ax.axvline(3.5, color='#c0392b', linestyle='-', lw=2)

fig_hist.suptitle('Counts across Competition Shocks',
               fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
unique_years = sorted(df_plot_data['year'].unique())
n_years = len(unique_years)




for r, year in enumerate(unique_years):
    # Histogram
    year_data = df_plot_data[df_plot_data['year'] == year]
    fig_hist, axes_hist = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    axes_hist = axes_hist.flatten()

    for i, (group_name, group_data) in enumerate(year_data.groupby('Comp_Group')):
        if group_name == 'Other':
            continue
        ax = axes_hist[i]
        sns.histplot(group_data['dynamic_rating'], ax=ax, kde=False, stat='density',
                    color=palette[group_name], edgecolor='black', bins=10)
        ax.set_title(group_name, size=13, fontweight='bold')
        ax.set_xlabel("Dynamic Rating")
        ax.set_ylabel("Density")
        ax.set_xlim(2.5, 5.1)

        # Add mean and 3.5 rating line
        mean_val = group_data['dynamic_rating'].mean()
        ax.axvline(mean_val, color='black', linestyle='--', lw=1.5)
        ax.text(mean_val*1.02, ax.get_ylim()[1]*0.8, f'Mean: {mean_val:.2f}',
                fontsize=11, fontweight='bold')
        ax.axvline(3.5, color='#c0392b', linestyle='-', lw=2)

    fig_hist.suptitle(f'{year} Counts across Competition Shocks',
                  fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# Test

In [ ]:
!pip install linearmodels

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from sklearn.neighbors import BallTree
import warnings
warnings.filterwarnings('ignore')

print("Build Panel Data")

coords_rad = np.radians(df_reg[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')

indices_5mi, dists_5mi = tree.query_radius(coords_rad, r=5.0 / 3958.8, return_distance=True)

panel_records = []
for i, row in df_reg.iterrows():
    entry_y = int(row['entry_year'])

    neighbor_indices = indices_5mi[i]
    neighbor_dists = dists_5mi[i] * 3958.8

    mask_self = (neighbor_indices != i)
    n_idx = neighbor_indices[mask_self]
    n_dist = neighbor_dists[mask_self]
    n_years = df_reg.iloc[n_idx]['entry_year'].values

    for current_year in range(entry_y, 2025):
        # Active, Shock
        mask_active = (n_years <= current_year)
        mask_shock = (n_years == (current_year - 1))

        active_dist = n_dist[mask_active]
        shock_dist = n_dist[mask_shock]

        # Gravity Model
        # weight：1 / (1 + dist)
        gravity_density = np.sum(1.0 / (1.0 + active_dist))
        gravity_shock = np.sum(1.0 / (1.0 + shock_dist))

        # KNN
        # 5th competiotor how far
        dist_5th_active = np.sort(active_dist)[4] if len(active_dist) >= 5 else 5.0
        dist_nearest_shock = np.min(shock_dist) if len(shock_dist) >= 1 else 5.0

        # Concentric Circles
        active_0_05 = np.sum(active_dist <= 0.5)
        active_05_2 = np.sum((active_dist > 0.5) & (active_dist <= 2.0))
        active_2_5  = np.sum((active_dist > 2.0) & (active_dist <= 5.0))
        shock_0_05 = np.sum(shock_dist <= 0.5)
        shock_05_2 = np.sum((shock_dist > 0.5) & (shock_dist <= 2.0))
        shock_2_5  = np.sum((shock_dist > 2.0) & (shock_dist <= 5.0))

        panel_records.append({
            'clinic_id': row['clinic_id'],
            'title': row['title'],
            'zip_clean': row['zip_clean'],
            'year': current_year,
            'clinic_age': current_year - entry_y,
            'distance_to_hub': row['distance_to_hub'],
            'mapped_location': row['mapped_location'],

            'log_gravity_density': np.log1p(gravity_density),
            'log_gravity_shock': np.log1p(gravity_shock),

            'log_dist_5th_active': np.log1p(dist_5th_active),
            'log_dist_nearest_shock': np.log1p(dist_nearest_shock),

            'log_density_0_05': np.log1p(active_0_05),
            'log_density_05_2': np.log1p(active_05_2),
            'log_density_2_5': np.log1p(active_2_5),
            'log_shock_0_05': np.log1p(shock_0_05),
            'log_shock_05_2': np.log1p(shock_05_2),
            'log_shock_2_5': np.log1p(shock_2_5)
        })

df_panel = pd.DataFrame(panel_records)


df_panel['zip_str'] = df_panel['zip_clean'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df_panel = df_panel.merge(final_yearly_stats, left_on=['title', 'zip_str', 'year'], right_on=['title', 'zip_str', 'review_year'], how='left')

df_panel['new_count'] = df_panel['new_count'].fillna(0)
df_panel['new_stars'] = df_panel['new_stars'].fillna(0)
df_panel = df_panel.sort_values(by=['clinic_id', 'year'])
df_panel['cumulative_votes'] = df_panel.groupby('clinic_id')['new_count'].cumsum()
df_panel['cumulative_stars'] = df_panel.groupby('clinic_id')['new_stars'].cumsum()

df_panel['dynamic_rating'] = np.where(df_panel['cumulative_votes'] > 0, df_panel['cumulative_stars'] / df_panel['cumulative_votes'], np.nan)
df_panel['log_votes_dynamic'] = np.log1p(df_panel['cumulative_votes'])

df_panel_valid = df_panel.dropna(subset=['dynamic_rating']).reset_index(drop=True)

def get_loc_type(loc):
    loc_str = str(loc)
    if loc_str.endswith('_S'): return 'Small'
    if loc_str.endswith('_M'): return 'Mid_Size'
    if loc_str.endswith('_L'): return 'Large'
    return 'Unknown'

df_panel_valid['Location_Type'] = df_panel_valid['mapped_location'].apply(get_loc_type)
df_panel_valid['Location_Type'] = pd.Categorical(df_panel_valid['Location_Type'], categories=['Mid_Size', 'Large', 'Small'], ordered=False)




In [ ]:
# OLS
if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])

controls = "+ clinic_age + log_votes_dynamic + EntityEffects"

# Gravity Model
print("="*60)
print("Model1: Gravity")
formula_gravity = f"dynamic_rating ~ log_gravity_shock + log_gravity_density {controls}"
mod_gravity = PanelOLS.from_formula(formula_gravity, data=df_panel_valid, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
print(mod_gravity.summary)

#Distance to Competitor
print("\n" + "="*60)
print("Model2: KNN")
formula_knn = f"dynamic_rating ~ log_dist_nearest_shock + log_dist_5th_active {controls}"
mod_knn = PanelOLS.from_formula(formula_knn, data=df_panel_valid, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
print(mod_knn.summary)

# Concentric Circles
print("\n" + "="*60)
print("Model3: Circle")
formula_concentric = f"dynamic_rating ~ log_shock_0_05 + log_shock_05_2 + log_shock_2_5 + log_density_0_05 + log_density_05_2 + log_density_2_5 {controls}"
mod_concentric = PanelOLS.from_formula(formula_concentric, data=df_panel_valid, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
print(mod_concentric.summary)

In [ ]:
# Concentric Circles
print("\n" + "="*60)
print("Model4: 0.5mile")
formula_concentric = f"dynamic_rating ~ log_shock_0_05 + log_density_0_05 {controls}"
mod_concentric = PanelOLS.from_formula(formula_concentric, data=df_panel_valid, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
print(mod_concentric.summary)

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Regressions by City Size")

if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])


formula_base = """
    dynamic_rating ~ log_shock_0_05 + log_shock_05_2 + log_shock_2_5
                   + log_density_0_05 + log_density_05_2 + log_density_2_5
                   + clinic_age + log_votes_dynamic + EntityEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{loc} City")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")


    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_base, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error")

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Regressions by City Size")

if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])


formula_base = """
    dynamic_rating ~ log_shock_0_05
                   + log_density_0_05
                   + clinic_age + log_votes_dynamic + EntityEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{loc} City")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")


    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_base, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error")

In [ ]:
import pandas as pd

spatial_vars = [
    'log_shock_0_05', 'log_shock_05_2', 'log_shock_2_5',
    'log_density_0_05', 'log_density_05_2', 'log_density_2_5'
]

corr_matrix = df_panel_valid[spatial_vars].corr().round(3)

print(corr_matrix.to_string())

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Entry Rate")

df_panel_valid['rate_shock_0_05'] = df_panel_valid['log_shock_0_05'] / (df_panel_valid['log_density_0_05'] + 1)
df_panel_valid['rate_shock_05_2'] = df_panel_valid['log_shock_05_2'] / (df_panel_valid['log_density_05_2'] + 1)
df_panel_valid['rate_shock_2_5']  = df_panel_valid['log_shock_2_5'] / (df_panel_valid['log_density_2_5'] + 1)


if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])

formula_base = """
    dynamic_rating ~ rate_shock_0_05 + rate_shock_05_2 + rate_shock_2_5
                   + log_density_0_05 + log_density_05_2 + log_density_2_5
                   + clinic_age + log_votes_dynamic + EntityEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{loc} City")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")


    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_base, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error")

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Entry Rate")

df_panel_valid['rate_shock_0_05'] = df_panel_valid['log_shock_0_05'] / (df_panel_valid['log_density_0_05'] + 1)



if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])

formula_base = """
    dynamic_rating ~ rate_shock_0_05
                   + log_density_0_05
                   + clinic_age + log_votes_dynamic + EntityEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{loc} City")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")


    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_base, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error")

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Regressions by City Size (Entity + Time Effects)")

if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])


formula_twfe = """
    dynamic_rating ~ rate_shock_0_05 + rate_shock_05_2 + rate_shock_2_5
                   + log_density_0_05 + log_density_05_2 + log_density_2_5
                   + log_votes_dynamic + EntityEffects + TimeEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")

    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_twfe, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

print("Regressions by City Size (Entity + Time Effects)")

if 'clinic_id' in df_panel_valid.columns and 'year' in df_panel_valid.columns:
    df_panel_valid = df_panel_valid.set_index(['clinic_id', 'year'])


formula_twfe = """
    dynamic_rating ~ rate_shock_0_05
                   + log_density_0_05
                   + log_votes_dynamic + EntityEffects + TimeEffects
"""

for loc in ['Large', 'Mid_Size', 'Small']:
    print(f"\n\n{'='*80}")
    print(f"{'='*80}")

    df_temp = df_panel_valid.reset_index()
    df_sub = df_temp[df_temp['Location_Type'] == loc]

    n_obs = len(df_sub)
    n_entities = df_sub['clinic_id'].nunique()
    print(f"Observations = {n_obs} | Solo = {n_entities}\n")

    if n_entities < 30:
        print("Too small for fixed effects")
        continue

    df_sub = df_sub.set_index(['clinic_id', 'year'])

    try:
        mod_sub = PanelOLS.from_formula(formula_twfe, data=df_sub, drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True)
        print(mod_sub.summary)
    except Exception as e:
        print(f"Error: {e}")